In [5]:
from ioMicro import *
import tifffile
import napari
# from shapely import Polygon,Point
import zarr,os
import numpy as np
from napari.viewer import Viewer
viewer = Viewer()

In [6]:
def read_im(path,return_pos=False):
    import zarr,os
    from dask import array as da
    dirname = os.path.dirname(path)
    fov = os.path.basename(path).split('_')[-1].split('.')[0]
    #print("Bogdan path:",path)
    file_ = dirname+os.sep+fov+os.sep+'data'
    #image = zarr.load(file_)[1:]
    image = da.from_zarr(file_)[1:]

    shape = image.shape
    #nchannels = 4
    xml_file = os.path.dirname(path)+os.sep+os.path.basename(path).split('.')[0]+'.xml'
    if os.path.exists(xml_file):
        txt = open(xml_file,'r').read()
        tag = '<z_offsets type="string">'
        zstack = txt.split(tag)[-1].split('</')[0]
        
        tag = '<stage_position type="custom">'
        x,y = eval(txt.split(tag)[-1].split('</')[0])
        
        nchannels = int(zstack.split(':')[-1])
        nzs = (shape[0]//nchannels)*nchannels
        image = image[:nzs].reshape([shape[0]//nchannels,nchannels,shape[-2],shape[-1]])
        image = image.swapaxes(0,1)
    shape = image.shape
    if image.dtype=='uint8':
        image = image.astype(np.float32)**2
    if return_pos:
        return image,x,y
    return image

### Napari

In [11]:
import napari
import numpy as np

fl4=r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\RNA\MYH6i\H4_NC_set2\Conv_zscan__076.zarr'
fl5=r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\RNA\MYH7i\H5_NC_set2\Conv_zscan__076.zarr'
fl3=r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\RNA\MYH6e_MYH7e_MYL2e\H3_NC_set2\Conv_zscan__076.zarr'
# #fl4=r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\A4_SC35_H2AX_set5\Conv_zscan5__115.zarr'
# # fl5=r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\A1_HTTm_NUP98_set5\Conv_zscan5__215.zarr'
# # fl6=r'M:\Jess\NeuronTracing\20260116_NeuronTracing_forebrain_PositiveControlTest\set1\Conv_zscan1_21.zarr'
# fl7=r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\A2_HTTr_TDP43_set5\Conv_zscan5__036.zarr'
im3=read_im(fl3)
im4=read_im(fl4)
im5=read_im(fl5)
# im4=read_im(fl4)

viewer = Viewer()
layer=viewer.add_image(im3)
layer=viewer.add_image(im4)
layer=viewer.add_image(im5)
# v.add_image(im4)
# v.add_image(DCBBMER)
#v.add_image(H1)
#v.add_image(set3)

### Check hybes

In [9]:
folders = [r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA']
hfolders = [fld_ for fld in folders for fld_ in glob.glob(fld+os.sep+'H*_NC_set2') ]
fov = 'Conv_zscan__076.zarr'

hnumbers = [get_iH(fld) for fld in hfolders]
hfolders = np.array(hfolders)[np.argsort(hnumbers)]
fls = [fld+os.sep+fov for fld in hfolders]
ims = []
for fl in tqdm(fls):
    # try:
    im = read_im(fl)
    ims.append(im[:,[20]])
    # except:
    #     print(fl)
    #     raise
from dask import array as da
imsf = da.concatenate(ims,axis=1)
import napari
viewer = Viewer()
layer = viewer.add_image(imsf)

100%|██████████| 42/42 [02:02<00:00,  2.91s/it]


In [7]:
folders = [r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA']
hfolders = [fld_ for fld in folders for fld_ in glob.glob(fld+os.sep+'H*_set2') ]
fov = 'Conv_zscan__076.zarr'

hnumbers = [get_iH(fld) for fld in hfolders]
hfolders = np.array(hfolders)[np.argsort(hnumbers)]
fls = [fld+os.sep+fov for fld in hfolders]
ims = []
for fl in tqdm(fls):
    # try:
    im = read_im(fl)
    h_max = da.asarray(im[0]).max(axis=0)
    ims.append(h_max)
    # except:
    #     print(fl)
    #     raise
from dask import array as da
imsf = da.concatenate(ims,axis=1)
import napari
viewer = Viewer()
layer = viewer.add_image(imsf)

0it [00:00, ?it/s]


ValueError: Need array(s) to concatenate

In [31]:
import imageio.v2 as imageio
import numpy as np
from PIL import Image, ImageDraw

gif_path = "H_max_projection_color0.gif"
hlabels = [get_iH(folder) for folder in hfolders]

with imageio.get_writer(gif_path, mode="I", duration=1000, loop=0) as writer:
    for hlabel, h_image in tqdm(zip(hlabels, ims), total=len(ims)):
        frame = np.asarray(h_image.compute())

        # Convert intensities to 8-bit for GIF display.
        low, high = np.percentile(frame, (1, 99.8))
        frame = np.clip((frame - low) / (high - low), 0, 1)
        frame = (frame * 255).astype(np.uint8)

        # Optional: shrink large images and label each frame.
        frame = Image.fromarray(frame)
        frame.thumbnail((768, 768))

        draw = ImageDraw.Draw(frame)
        draw.text((12, 12), f"H{hlabel}", fill=255)

        writer.append_data(np.asarray(frame))

print(f"Saved: {gif_path}")

100%|██████████| 22/22 [05:49<00:00, 15.88s/it]

Saved: H_max_projection_color0.gif


### Make mosaic

In [10]:
def get_mosaic_image(data_fld, resc=4, icol=1, frame=20, force=False, rescz=2):
    save_fld = os.path.dirname(data_fld)+os.sep+'mosaics_test'   # Save to folder
    if not os.path.exists(save_fld): os.makedirs(save_fld)
    fl_save = save_fld+os.sep+os.path.basename(data_fld)+'_frame'+str(frame)+'_col'+str(icol)+'.tiff'
    print("Will save to file:", fl_save)
    
    fls_ = np.sort(glob.glob(data_fld+r'\*.zarr'))
    ims, xs_um, ys_um = [], [], []
    
    for fl in tqdm(fls_[:]):
        try:
            im, x, y = read_im(fl, return_pos=True)
            
            if str(frame).lower() == 'all':
                ims.append(np.array(np.max(im[icol][::rescz,::resc,::resc][:, ::-1], axis=0), dtype=np.float32))
            else:
                ims.append(np.array(im[icol][frame,::resc,::resc][:, ::-1], dtype=np.float32))
            xs_um.append(x)
            ys_um.append(y)
        except Exception as e:
            print(f"Error processing file {fl}: {str(e)}")
            continue
    
    if not ims:
        print("No images were successfully processed. Exiting.")
        return
    
    # Continue with the rest of the function as before
    im_big, xs, ys = compose_mosaic([im.T[::-1,::-1] for im in ims], xs_um, ys_um, ims_c=None,
                                  um_per_pix=0.1083333*resc,
                                  rot=90, return_coords=True) # Rotate each FOV
    
    import tifffile
    tifffile.imwrite(fl_save, im_big)
    
    resc_ = 3
    data_fld = os.path.dirname(fls_[0])
    fig = plt.figure(figsize=(30,30))
    im__ = im_big[::resc_,::resc_]
    vmax = np.percentile(im__[im__>0], 99.)
    vmin = np.percentile(im__[im__>0], 1.)
    plt.imshow(im_big.T[::resc_,::resc_], vmin=vmin, vmax=vmax, cmap='gray')
    for x_, y_, fl_ in zip(xs, ys, fls_):
        ifov = fl_.split('_')[-1].split('.')[0]
        plt.text(x_/resc_, y_/resc_, ifov, color='r')
    plt.xticks([])
    plt.yticks([])
    fl_save = fl_save.replace('.tiff', '_annot.png')
    print(fl_save)
    fig.savefig(fl_save)
    plt.close('all')

def get_mosaic_image_mahsa(data_fld,resc = 4,icol = 1,frame = 20,force=False,rescz=2):
    save_fld = os.path.dirname(data_fld)+os.sep+'mosaics'     # Save to folder
    if not os.path.exists(save_fld): os.makedirs(save_fld)
    fl_save = save_fld+os.sep+os.path.basename(data_fld)+'_frame'+str(frame)+'_col'+str(icol)+'.tiff'
    print("Will save to file:",fl_save)
    if not os.path.exists(fl_save) or force:
        fls_ = np.sort(glob.glob(data_fld+r'\*.zarr'))#[:50]
        ims,xs_um,ys_um=[],[],[]

        for fl in tqdm(fls_[:]):
            im,x,y = read_im(fl,return_pos=True)
            
            if str(frame).lower()=='all':
                im =np.array(np.max(im[icol][::resc,::resc,::resc],axis=0),dtype=np.float32)
            # elif type(sl) is slice:
            #     im =np.array(im[icol][frame,::resc,::resc],dtype=np.float32)
            else:
                im = np.array(im[icol][frame,::resc,::resc],dtype=np.float32)
            # Rotate -90 degrees (counter-clockwise) and mirror along Y-axis
            im = np.flipud(np.fliplr(im))  # rotate -90°, then flip up-down np.rot90(im, k=1)
            ims.append(np.swapaxes(im,-1,-2)[...,::-1,::-1])
            xs_um.append(x)
            ys_um.append(y)
        im_big,xs,ys = compose_mosaic(ims,xs_um,ys_um,ims_c=None,
                                      um_per_pix=0.11*resc,
                                      rot = 0,return_coords= True)#-90 to 0a
       # for fl in tqdm(fls_[:]):
        #     im,x,y = read_im(fl,return_pos=True)
            
        #     if str(frame).lower()=='all':
        #         ims.append(np.array(np.max(im[icol][::rescz,::resc,::resc][:, ::-1],axis=0),dtype=np.float32))
        #     else:
        #         ims.append(np.array(im[icol][frame,::resc,::resc][:, ::-1],dtype=np.float32))
        #     xs_um.append(x)
        #     ys_um.append(y)

        # im_big,xs,ys = compose_mosaic([im.T[::-1,::-1] for im in ims],xs_um,ys_um,ims_c=None,
                                      # um_per_pix=0.1083333*resc,
                                      # rot = 0,return_coords= True)
        
        import tifffile
        tifffile.imwrite(fl_save,im_big)
        
        
        resc_ = 3
        data_fld = os.path.dirname(fls_[0])
        fig = plt.figure(figsize=(30,30))
        im__ = im_big[::resc_,::resc_]
        #if vmax is None:
        vmax = np.percentile(im__[im__>0],99.)
        vmin=np.percentile(im__[im__>0],1.)
        plt.imshow(im_big.T[::resc_,::resc_],vmin=vmin,vmax=vmax,cmap='gray')
        #fig.savefig(fl_save)
        for x_,y_,fl_ in zip(xs,ys,fls_):
            ifov = fl_.split('_')[-1].split('.')[0]
            plt.text(x_/resc_,y_/resc_,ifov,color='r')
        plt.xticks([])
        plt.yticks([])
        fl_save = fl_save.replace('.tiff','_annot.png')
        print(fl_save)
        fig.savefig(fl_save)
        plt.close('all')

In [14]:
cols = [0,1]
# sets=[1,2]
for set_num in range(1,13): 
    for col in cols:
        data_fld = r'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set' + str(set_num)
        print(f"\nProcessing {data_fld} with col={col}")
        get_mosaic_image_mahsa(data_fld, resc=5, icol=col, frame=20, force=False, rescz=2)



Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set1 with col=0
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set1_frame20_col0.tiff

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set1 with col=1
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set1_frame20_col1.tiff

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set2 with col=0
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set2_frame20_col0.tiff

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set2 with col=1
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set2_frame20_col1.tiff

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set3 with col=0
Will save to file: \\merfish13\Public\F

100%|██████████| 154/154 [54:23<00:00, 21.19s/it] 


\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set3_frame20_col0_annot.png

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set3 with col=1
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set3_frame20_col1.tiff


100%|██████████| 154/154 [50:11<00:00, 19.55s/it] 


\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set3_frame20_col1_annot.png

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set4 with col=0
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set4_frame20_col0.tiff


100%|██████████| 144/144 [32:45<00:00, 13.65s/it]


... (31 outputs left out)

100%|██████████| 113/113 [01:53<00:00,  1.00s/it]


\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set12_frame20_col0_annot.png

Processing \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\H42_NC_set12 with col=1
Will save to file: \\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set12_frame20_col1.tiff


100%|██████████| 113/113 [01:39<00:00,  1.13it/s]


\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set12_frame20_col1_annot.png


In [22]:
import numpy as np
import os
import glob
from tqdm import tqdm
sets = [5]
cols = [0,1]

for set_num in sets:
    for col in cols:
        data_fld = r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\H5_BICAN_set' + str(set_num)
        print(f"\nProcessing {data_fld} with col={col}")
        get_mosaic_image_mahsa(data_fld, resc=2, icol=col, frame='all', force=True, rescz=2)


Processing Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\H5_BICAN_set5 with col=0
Will save to file: Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\mosaics\H5_BICAN_set5_frameall_col0.tiff


100%|████████████████████████████████████████████████████████████████████████████████| 988/988 [54:43<00:00,  3.32s/it]


Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\mosaics\H5_BICAN_set5_frameall_col0_annot.png

Processing Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\H5_BICAN_set5 with col=1
Will save to file: Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\mosaics\H5_BICAN_set5_frameall_col1.tiff


 19%|███████████████▏                                                                | 188/988 [10:27<44:28,  3.34s/it]


KeyboardInterrupt: 

### Look at tiff

In [24]:
im0 = tifffile.imread(r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\mosaics\H5_BICAN_set5_frameall_col0.tiff')
im1 = tifffile.imread(r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\BICAN_MERFISH\mosaics\H10_BICAN_set5_frameall_col0.tiff')
#im2 = tifffile.imread(r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\mosaics\A1_HTTm_NUP98_set1_frame20_col-2.tiff')
# dapi = tifffile.imread(r'Z:\Yuanyi\HDHuman_Striatum_BICAN_12_16_25\mosaics\A2_HTTr_TDP43_set6_frame20_col3.tiff')
V= napari.view_image(im0,contrast_limits=[5000,25000], colormap = 'yellow', blending = 'Additive', name='5')
V.add_image(im1,contrast_limits=[0,1000000], colormap = 'red', blending = 'Additive', name = '10')
#V.add_image(im2,contrast_limits=[0,100000], colormap = 'orange', blending = 'Additive', name= '3')
# V.add_image(dapi,contrast_limits=[3000,350000], colormap = 'bop blue', blending = 'Additive', name= 'DAPI')

c:\Users\Scope7\anaconda3\envs\cellpose\lib\site-packages\napari\_vispy\layers\scalar_field.py:198: UserWarning: data shape (33773, 77273) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
c:\Users\Scope7\anaconda3\envs\cellpose\lib\site-packages\napari\_vispy\layers\scalar_field.py:198: UserWarning: data shape (33773, 77272) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer '10' at 0x11a67a87dc0>

c:\Users\Scope7\anaconda3\envs\cellpose\lib\site-packages\napari\_vispy\layers\scalar_field.py:198: UserWarning: data shape (33773, 77273) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


In [15]:
import napari
import tifffile

viewer = napari.Viewer()

for i in range(1, 13):

    path = rf'\\merfish13\Public\Fugui_Jess\Myh67_mouse_Kiwi_09_21_2026\DNA\mosaics\H42_NC_set{i}_frame20_col0.tiff'

    im = tifffile.imread(path)

    viewer.add_image(
        im,
        contrast_limits=[0, 10000],
        name=f"H1_set{i}"
    )

napari.run()